# 01 - CWRU Bearing Data Exploration

This notebook is a read-only inspection of the CWRU Bearing Data Center MATLAB files that have been dropped into `ml/data/raw/cwru/`.

It intentionally performs **no** windowing, feature extraction, or training. Its purpose is to confirm that a downloaded `.mat` file:

1. Opens with `scipy.io.loadmat`.
2. Contains a drive-end time-domain vibration variable (`*_DE_time`).
3. Optionally contains an `RPM` variable.

If no files are present the notebook prints a clear instruction rather than raising an obscure error.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'ml').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'ml' / 'data' / 'raw' / 'cwru'
RAW_DIR

In [ ]:
mat_files = sorted(RAW_DIR.glob('*.mat'))
if not mat_files:
    print(
        f"No .mat files found in {RAW_DIR}.\n"
        "Download the initial 16 recordings from https://engineering.case.edu/bearingdatacenter\n"
        "(12 kHz Drive End, fault size 0.007\" for Normal/IR/B/OR@6) and drop the .mat files here."
    )
else:
    for p in mat_files:
        print(p.name)

## Inspect one file

Pick the first `.mat` file we found and describe it.

In [ ]:
from ml.src.cwru_loader import inspect_mat, load_recording

if not mat_files:
    raise SystemExit('No .mat files available yet - see the cell above for download instructions.')

target = mat_files[0]
summary = inspect_mat(target)
for k, v in summary.items():
    print(f'{k}: {v}')

In [ ]:
recording = load_recording(target)
signal = recording.drive_end_signal
print(f'Drive-end variable: {recording.drive_end_key}')
print(f'Samples: {signal.size}')
print(f'RPM (measured): {recording.rpm}')
print(f'min={signal.min():.4f}  max={signal.max():.4f}  mean={signal.mean():.4f}  std={signal.std():.4f}')

In [ ]:
import matplotlib.pyplot as plt

preview_n = min(5000, signal.size)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(signal[:preview_n], linewidth=0.5)
ax.set_xlabel('sample index')
ax.set_ylabel('acceleration (a.u.)')
ax.set_title(f'{target.name} - first {preview_n} samples ({recording.drive_end_key})')
plt.tight_layout()
plt.show()